In [3]:
import json
import joblib
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Load dataset
with open("medical_intents.json", encoding="utf-8") as file:
    data = json.load(file)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z ]", "", text)
    return text

X = []
y = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        X.append(clean_text(pattern))
        y.append(intent["tag"])

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Vectorization
vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_vec, y_encoded)

# Save model components
joblib.dump(model, "chatbot_model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")

print("✅ Medical chatbot trained successfully")


✅ Medical chatbot trained successfully


In [ ]:
import json
import random
import joblib
import re

model = joblib.load("chatbot_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")
label_encoder = joblib.load("label_encoder.pkl")

with open("medical_intents.json", encoding="utf-8") as file:
    data = json.load(file)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z ]", "", text)
    return text

def medical_chatbot(user_input):
    user_input = clean_text(user_input)
    X_test = vectorizer.transform([user_input])
    
    probs = model.predict_proba(X_test)[0]
    confidence = max(probs)

    if confidence < 0.2:
        return "I'm not sure. Please consult a medical professional."

    intent_index = probs.argmax()
    intent_tag = label_encoder.inverse_transform([intent_index])[0]

    for intent in data["intents"]:
        if intent["tag"] == intent_tag:
            return random.choice(intent["responses"])

print("🏥 Medical Chatbot is running (type 'exit' to stop)")

while True:
    user = input("You: ")
    if user.lower() == "exit":
        print("Bot: Take care and stay healthy!")
        break
    print("Bot:", medical_chatbot(user))


🏥 Medical Chatbot is running (type 'exit' to stop)


You:  i have fever


Bot: Fever may indicate an infection. Please stay hydrated and rest.


You:  and head pain


Bot: Try resting and drinking water. Consult a doctor if it continues.


You:  also Sneezing


Bot: If symptoms worsen, consult a doctor.
